# 02 — Features and the rules baseline
The difference features from the brief (diff size/sign, ratio to amount, period-boundary flag, missing-in-new flag, plan type), and what the hand-written rules already explain before any ML is involved.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from src.generate_data import generate
from src.features import build_features, build_ledger_features
from src.rules import classify_rules

deals = generate(6000, seed=42)
diffs = deals[deals["new_commission"] != deals["old_commission"]].copy()
X = build_features(diffs)
X.head(6)

,diff,abs_diff,new_over_old,amount_ratio,amount_missing_new,crosses_period,new_is_zero,is_cent_level,amount_old,plan_Standard,plan_Tiered,plan_Strategic
7,-1089.05,1089.05,0.000000,1.000000,0,1,1,0,21780.96,1,0,0
14,-440.52,440.52,0.000000,1.000000,0,1,1,0,8810.50,1,0,0
22,-53.19,53.19,0.833344,1.000000,0,0,0,0,5319.36,0,1,0
23,85.70,85.70,1.151341,1.151347,0,0,0,0,11325.35,1,0,0
26,0.01,0.01,1.000020,1.000000,0,0,0,1,10239.94,1,0,0
29,-55.11,55.11,0.749989,1.000000,0,0,0,0,5510.73,0,0,1


## Which signals separate the causes?
Mean feature values by injected cause — the patterns a model (or a reviewer) has to work with:

In [3]:
X.join(diffs["cause"]).groupby("cause").mean().round(3)

,diff,abs_diff,new_over_old,amount_ratio,amount_missing_new,crosses_period,new_is_zero,is_cent_level,amount_old,plan_Standard,plan_Tiered,plan_Strategic
cause,,,,,,,,,,,,
data,-225.298,278.005,0.688,0.713,0.267,0.099,0.272,0.000,14899.974,0.470,0.322,0.208
rounding,-3.615,3.878,0.994,0.998,0.003,0.135,0.006,0.986,13683.676,0.464,0.309,0.227
rule,-137.445,138.158,0.799,0.999,0.000,0.099,0.000,0.007,13925.405,0.493,0.270,0.238
timing,-708.677,708.677,0.016,1.000,0.000,0.982,0.982,0.009,13909.411,0.500,0.312,0.188


## Rules baseline, no split — just the mechanics
Ordered checks: exact match → none; missing record → data; new = 0 across a period boundary → timing; |diff| ≤ $0.05 → rounding; amount changed → data; anything else → rule.

In [4]:
pred = classify_rules(diffs)
pd.crosstab(diffs["cause"], pred, rownames=["injected"], colnames=["rules say"])

rules say,data,rounding,rule,timing
injected,,,,
data,201,0,0,1
rounding,2,357,2,1
rule,2,2,278,0
timing,0,2,2,220


The rules are wrong in instructive places: dual-cause rows (a data change that also carries a rate change reads as pure `data` to the amount check) and the 2% of rows with noisy reviewer labels. That residual is the only place ML can earn anything.

In [5]:
from src.generate_ledger import generate_ledger
ledger = generate_ledger(seed=42)
Xl = build_ledger_features(ledger)
corr = Xl.assign(mis=ledger["is_mismapped"]).corr()["mis"].drop("mis").sort_values(key=abs, ascending=False)
print("Feature correlation with is_mismapped:")
corr.round(3)

Feature correlation with is_mismapped:


category_match      -0.882
name_similarity     -0.747
collision_count      0.508
abs_log_ratio        0.407
balance_zscore       0.370
balance_ratio        0.361
balance_magnitude   -0.080
sign_flip              NaN
zero_new               NaN
Name: mis, dtype: float64

**Takeaway.** Name similarity and category match are the loudest mapping signals, but migration renames (~30% of accounts) keep them from being oracles — the balance features (ratio, z-score vs the account's own history) have to carry weight too.